# Buddy — custom wake-word training

Trains an **openWakeWord** model for "hey buddy" and its variants, entirely from
**synthetic speech** (Piper TTS) — no recordings of Kai's voice needed, and
nothing leaves the machine at inference: the result is a ~1MB ONNX that runs
locally at a few % CPU.

**Runtime: GPU.** Runtime → Change runtime type → T4.

### Which phrases wake it (decided 2026-08-02)

Carrier forms only — `hey buddy`, `hi buddy`, `hello buddy`, `okay buddy`.
Three syllables with crisp /b/ /d/ plosives is the detection sweet spot, and
several carriers make the model robust to however you happen to say it.

**Bare `buddy` is trained as a NEGATIVE, on purpose.** It isn't merely left out:
a 2-syllable bare word would subsume the carriers entirely — once the model
fires on "buddy" alone, "hey" stops mattering and every conversational "buddy"
in earshot becomes a wake event. Listing it as a hard negative teaches the
model that the carrier is *required*, which is a sharper boundary than silence.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "NO GPU — Runtime → Change runtime type → T4, then re-run"
print(torch.cuda.get_device_name(0))

## 2. Dependencies

`openwakeword.train` needs a few extras beyond the inference install, and Piper's
sample generator is a repo rather than a package.

In [ ]:
# Pin the generator to v2.0.0. This is load-bearing, not caution:
#   • v2.0.0 has generate_samples.py at the root AND vendors piper_train/
#   • current master was restructured into a module with no generate_samples.py
#   • the PyPI package has the module layout and expects piper_train separately
# Both of the other routes fail — one with "no such file", the other with
# "No module named 'piper_train'". openWakeWord's own trainer expects v2.0.0.
!rm -rf /content/piper-sample-generator
!git clone -q --branch v2.0.0 --depth 1 \
  https://github.com/rhasspy/piper-sample-generator /content/piper-sample-generator
!mkdir -p /content/piper-sample-generator/models
!wget -q -O /content/piper-sample-generator/models/en_US-libritts_r-medium.pt \
  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt

!pip install -q piper-phonemize webrtcvad
!pip install -q openwakeword torchinfo torchmetrics onnx onnx_tf mutagen speechbrain
!pip install -q datasets soundfile librosa

# Verify the layout AND that it runs, before synthesising thousands of clips.
import os, subprocess, sys
GEN = "/content/piper-sample-generator"
assert os.path.exists(f"{GEN}/generate_samples.py"), "wrong layout: expected v2.0.0"
assert os.path.isdir(f"{GEN}/piper_train"), "piper_train not vendored: wrong tag"
MODEL = f"{GEN}/models/en_US-libritts_r-medium.pt"
size = os.path.getsize(MODEL) if os.path.exists(MODEL) else 0
assert size > 1_000_000, f"voice checkpoint missing/truncated ({size} bytes)"
probe = subprocess.run([sys.executable, "generate_samples.py", "--help"],
                       cwd=GEN, capture_output=True, text=True)
assert probe.returncode == 0, f"generator not runnable:\n{probe.stderr[-900:]}"
print(f"generator ready · voice checkpoint {size/1e6:.0f} MB")

## 3. Configure the phrases

In [ ]:
# --- what wakes Buddy -------------------------------------------------------
PHRASES = ["hey buddy", "hi buddy", "hello buddy", "okay buddy"]

# --- what must NOT wake it --------------------------------------------------
# Phonetic neighbours are what actually cause false fires; generic conversation
# is covered by the large negative corpus in step 5. Bare "buddy" leads this
# list deliberately — see the note at the top.
HARD_NEGATIVES = [
    "buddy", "buddy",                      # weighted: the carrier is required
    "body", "study", "muddy", "buttery", "bloody", "budding", "buddies",
    "butter", "bunny", "buggy", "already", "everybody", "somebody",
    "hey buzz", "hey bunny", "hi body", "hey there",
    "my buddy", "your buddy", "the buddy system", "hey buddy's car",
]

N_POSITIVE = 4000           # synthetic clips across the phrase set
N_HARD_NEG = 3000           # more than before: bare "buddy" needs the weight
print(f"positives:      {PHRASES}")
print(f"hard negatives: {len(HARD_NEGATIVES)} phrases (bare 'buddy' included)")

## 4. Synthesise the clips

Piper generates each phrase across many speakers, speeds and pitches — that
variation is what lets a model trained on synthetic speech generalise to a real
voice.

In [ ]:
import os, glob, subprocess, sys

GEN = "/content/piper-sample-generator"
MODEL = "models/en_US-libritts_r-medium.pt"   # relative — piper_train resolves from GEN

def synth(phrases, out_root, n_total):
    """Synthesise each phrase, failing loudly on empty output. An early version
    printed a success line unconditionally and reported '1000 clips' while every
    call had in fact failed."""
    os.makedirs(out_root, exist_ok=True)
    per = max(1, n_total // len(phrases))
    total = 0
    for i, phrase in enumerate(phrases):
        out = f"{out_root}/p{i}"
        os.makedirs(out, exist_ok=True)
        run = subprocess.run(
            [sys.executable, "generate_samples.py", phrase,
             "--model", MODEL, "--max-samples", str(per),
             "--batch-size", "64", "--output-dir", out],
            cwd=GEN, capture_output=True, text=True)
        made = len(glob.glob(f"{out}/*.wav"))
        if made == 0:
            raise RuntimeError(
                f"{phrase!r} produced no audio (exit {run.returncode})\n"
                f"--- stderr ---\n{run.stderr[-1500:]}")
        total += made
        print(f"  {phrase!r:24} → {made} clips")
    return total

pos_n = synth(PHRASES, "/content/positive", N_POSITIVE)
neg_n = synth(HARD_NEGATIVES, "/content/negative_hard", N_HARD_NEG)
print(f"\npositives {pos_n} · hard negatives {neg_n}")

## 5. Background negatives + room augmentation

openWakeWord ships pre-computed features from a large speech corpus — this is
what teaches the model that ordinary conversation is *not* the wake word. The
impulse responses simulate real rooms so the model survives being spoken to from
across a desk.

In [ ]:
!mkdir -p /content/oww_data && cd /content/oww_data && \
  wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy && \
  wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
!ls -lh /content/oww_data

## 6. Train

In [ ]:
from openwakeword.train import Model as TrainModel

model = TrainModel(n_classes=1, input_shape=(16, 96), model_type="dnn",
                   layer_dim=128, seconds_per_example=1.44)

model.auto_train(
    positive_dir="/content/positive",
    negative_dirs=["/content/negative_hard"],
    background_features=["/content/oww_data/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"],
    validation_features="/content/oww_data/validation_set_features.npy",
    steps=50_000,
    target_false_positives_per_hour=0.5,   # the knob that trades sensitivity for calm
)

## 7. Validate — separation is the number that matters

A model that scores 0.99 on the phrase is useless if it also scores 0.8 on
"study". Check the **gap**, and set the app's threshold inside it.

In [ ]:
import numpy as np, glob, soundfile as sf
from openwakeword.model import Model as OWWModel

model.export_to_onnx("/content/hey_buddy.onnx")
oww = OWWModel(wakeword_models=["/content/hey_buddy.onnx"])
key = list(oww.models.keys())[0]

def score_dir(pattern, limit=200):
    out = []
    for path in glob.glob(pattern, recursive=True)[:limit]:
        audio, _ = sf.read(path, dtype="int16")
        oww.reset()
        out.append(max(oww.predict(audio[i:i+1280])[key]
                       for i in range(0, len(audio) - 1280, 1280)))
    return np.array(out)

pos = score_dir("/content/positive/**/*.wav")
neg = score_dir("/content/negative_hard/**/*.wav")
bare = score_dir("/content/negative_hard/p0/**/*.wav")   # bare "buddy" only

print(f"positives   mean {pos.mean():.3f}   p05 {np.percentile(pos, 5):.3f}")
print(f"hard negs   mean {neg.mean():.3f}   p95 {np.percentile(neg, 95):.3f}")
print(f"bare buddy  mean {bare.mean():.3f}  p95 {np.percentile(bare, 95):.3f}  <- must stay LOW")

gap = np.percentile(pos, 5) - np.percentile(neg, 95)
print(f"\nseparation: {gap:+.3f}  ({'usable' if gap > 0.2 else 'TOO TIGHT — train longer or add negatives'})")
print(f"suggested threshold: {(np.percentile(pos, 5) + np.percentile(neg, 95)) / 2:.2f}")
if np.percentile(bare, 95) > 0.4:
    print("\n⚠️  bare 'buddy' still scores high — the carrier isn't being required.")
    print("    Add more bare-'buddy' negatives and retrain before shipping.")

## 8. Export

Download `hey_buddy.onnx`, drop it in `src/voice/` of the repo, and point the
listener at it:

```python
# src/menubar/app.py — where WakeWordListener is constructed
WakeWordListener(..., model_name=str(resource_path("src", "voice", "hey_buddy.onnx")))
```

Then set `features.wake_word: true` in config and rebuild. Use the suggested
threshold from step 7 as `WakeWordListener(threshold=…)`; raise it if it fires
in conversation, lower it if it misses you.

In [ ]:
from google.colab import files
files.download("/content/hey_buddy.onnx")